In [5]:
model_id = "YuWangX/memoryllm-8b"

In [6]:
import os

os.environ.get('HF_ENDPOINT')

'https://hf-mirror.com'

In [7]:
import torch

/home/ccm/miniconda3/envs/memoryllm/lib/python3.12/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/ccm/miniconda3/envs/memoryllm/lib/python3.12/site-packages/ipykerne

In [8]:
from modeling_memoryllm import MemoryLLM
from configuration_memoryllm import MemoryLLMConfig
from transformers import AutoTokenizer
# config = MemoryLLMConfig.from_pretrained(model_id)
model = MemoryLLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/home/ccm/miniconda3/envs/memoryllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - I

Memory Pool Parameters: 1.6777 B


Loading checkpoint shards: 100%|██████████| 8/8 [00:02<00:00,  3.14it/s]


In [18]:
model.initialized = torch.tensor(0, dtype=torch.uint8)

In [19]:
print(model.initialized)

tensor(0, dtype=torch.uint8)


In [10]:
device = "cuda:1"

In [11]:
model = model.to(device)

In [12]:

# Self-Update with the new context
ctx = "Last week, John had a wonderful picnic with David. During their conversation, David mentioned multiple times that he likes eating apples. Though he didn't mention any other fruits, John says he can infer that David also like bananas."

# please make sure the context to inject into the memory is larger than 16 tokens, this is the hard minimum when training the model. The memory will be disturbed when less than 16 tokens are injected into the memory. 
model.inject_memory(tokenizer(ctx, return_tensors='pt', add_special_tokens=False).input_ids.to(device), update_memory=True)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


tensor([[[[ 2.2705e-02,  6.3171e-03, -3.7537e-03,  ...,  3.4668e-02,
            2.7832e-02,  6.3477e-03],
          [-3.5400e-03,  2.9663e-02, -3.7842e-02,  ...,  1.7822e-02,
            1.6113e-02, -7.3242e-03],
          [-1.8066e-02,  3.9368e-03,  1.2451e-02,  ..., -4.2480e-02,
            2.8381e-03,  2.5635e-02],
          ...,
          [-3.2959e-03, -5.0659e-03, -1.1230e-02,  ..., -1.0742e-02,
            3.8300e-03, -1.6113e-02],
          [ 3.5156e-02,  4.4250e-03,  1.8188e-02,  ..., -5.0964e-03,
           -6.5918e-03, -1.3123e-03],
          [-8.2397e-04, -1.0498e-02, -3.4180e-03,  ..., -3.0518e-03,
           -3.5400e-03,  1.3062e-02]],

         [[ 1.4038e-02, -8.5449e-03, -1.9653e-02,  ..., -4.0771e-02,
            2.1118e-02,  1.2634e-02],
          [-3.1586e-03,  2.5635e-02, -1.3184e-02,  ..., -2.0752e-02,
            4.0527e-02,  1.1841e-02],
          [ 1.4038e-02,  6.3477e-03, -1.3550e-02,  ..., -6.6406e-02,
            4.2480e-02,  2.9297e-02],
          ...,
     

In [10]:
messages = [{
    'role': 'user', "content": "What fruits does David like?",
}]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)[:, 1:] # remove bos tokens as the model has its own trained bos embeddings.
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(input_ids=inputs.to(device),
                         max_new_tokens=20,
                         eos_token_id=terminators)

response = tokenizer.decode(outputs[0])

What fruits does David like? Answer:.?

..
..so


In [38]:
ctx = "Last week, John had a wonderful picnic with David. During their conversation, David mentioned multiple times that he likes eating apples. Though he didn't mention any other fruits, John says he can infer that David also like bananas."

inputs = tokenizer(ctx, return_tensors='pt', add_special_tokens=False)

print(inputs)

{'input_ids': tensor([[ 5966,  2046,    11,  3842,  1047,   264, 11364, 55562,   449,  6941,
            13, 12220,   872, 10652,    11,  6941,  9932,  5361,  3115,   430,
           568, 13452, 12459, 41776,    13, 18056,   568,  3287,   956,  6420,
           904,  1023, 26390,    11,  3842,  2795,   568,   649, 24499,   430,
          6941,  1101,  1093, 68442,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [25]:
input_ids = inputs.input_ids
input_ids = input_ids.to(device)
print(input_ids.device)

cuda:1


In [26]:
model.inject_memory(input_ids, update_memory=True)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


tensor([[[[ 1.6846e-02, -1.1414e-02,  1.9165e-02,  ..., -3.6377e-02,
            1.4282e-02, -2.8076e-02],
          [-3.9368e-03,  3.5400e-02,  2.5635e-02,  ..., -1.3184e-02,
           -4.9133e-03, -1.8066e-02],
          [-1.0071e-03, -1.5076e-02,  1.9836e-03,  ...,  4.3701e-02,
            8.7280e-03, -8.4839e-03],
          ...,
          [-3.0975e-03,  1.5259e-05, -8.9111e-03,  ..., -9.8267e-03,
            1.5717e-03, -2.7588e-02],
          [ 2.2339e-02, -4.7302e-04,  2.3193e-02,  ..., -1.4038e-02,
           -7.4158e-03, -9.3994e-03],
          [-4.5776e-03, -2.1362e-03,  3.0823e-03,  ..., -1.0803e-02,
           -5.4626e-03,  8.6670e-03]],

         [[-1.5869e-02,  2.7344e-01, -1.1475e-01,  ...,  5.0781e-01,
            3.7305e-01, -1.2158e-01],
          [ 2.7954e-02,  3.7079e-03, -3.1433e-03,  ..., -1.2988e-01,
           -3.0273e-02, -2.3926e-02],
          [ 2.2461e-02, -3.3203e-02,  1.9165e-02,  ..., -1.8433e-02,
            4.2236e-02, -2.2583e-03],
          ...,
     

In [14]:
# Generation
messages = [{
    'role': 'user', "content": "What fruits does David like?",
}]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)[:, 1:] # remove bos tokens as the model has its own trained bos embeddings.
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(input_ids=inputs.to(device),
                         max_new_tokens=100,
                         eos_token_id=terminators)

response = tokenizer.decode(outputs[0])
print(response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


<|start_header_id|>user<|end_header_id|>

What fruits does David like?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The city of New York is the most populous city in the United States and the most populous city in the United States of America, with a population of 8,622,000 in 2020. The city's population is 8,622,000, the population of New York City is 8,622,000. The city's population is 8,622,000, the population of New York City is 8,622,000. The city's population is 8


In [22]:
inputs = tokenizer("Question: What fruits does David like? Answer: ", return_tensors='pt', add_special_tokens=False)
print(inputs.to(device))

outputs = model.generate(**inputs, max_new_tokens=20)

input_len = inputs.input_ids.shape[1]   # 或者 inputs['input_ids'].shape[1]

response = tokenizer.decode(outputs[0][input_len:])
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'input_ids': tensor([[14924,    25,  3639, 26390,  1587,  6941,  1093,    30, 22559,    25,
           220]], device='cuda:1'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:1')}
4
Answer: a) Fruits
Answer: b) Apples
Answer: c)
